# 🧪 Paddle wheel diagnostic

Ejecutá esta única celda **antes de descargar nuevamente**.

No descarga el wheel entero: solo compara el archivo persistente con
dos pequeños rangos HTTP del archivo oficial.

In [ ]:
import pathlib, subprocess, shutil, re, binascii, os, sys

URL = (
    "https://paddle-whl.cdn.bcebos.com/stable/cu126/paddlepaddle-gpu/"
    "paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl"
)

WHEEL = pathlib.Path.home() / ".cache" / "paddle-wheels" / \
    "paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl"

def hb(data):
    return binascii.hexlify(data).decode("ascii")

def ascii_preview(data):
    return "".join(chr(b) if 32 <= b < 127 else "." for b in data)

def curl_range(range_spec):
    curl = shutil.which("curl")
    if not curl:
        raise RuntimeError("curl no está instalado.")

    header = pathlib.Path("/tmp/paddle_range_headers.txt")
    body = pathlib.Path("/tmp/paddle_range_body.bin")

    for p in (header, body):
        if p.exists():
            p.unlink()

    p = subprocess.run(
        [
            curl,
            "--location",
            "--silent",
            "--show-error",
            "--fail",
            "--range", range_spec,
            "--dump-header", str(header),
            "--output", str(body),
            URL,
        ],
        capture_output=True,
        text=True,
    )

    headers = header.read_text(errors="replace") if header.exists() else ""
    data = body.read_bytes() if body.exists() else b""
    return p.returncode, headers, data, p.stderr

print("🧪 DIAGNÓSTICO DEL WHEEL EXISTENTE")
print("NO descarga el wheel completo.")
print()

if not WHEEL.exists():
    raise RuntimeError(f"No existe: {WHEEL}")

size = WHEEL.stat().st_size
print("Archivo:", WHEEL)
print("Tamaño local:", size, "bytes")
print()

with WHEEL.open("rb") as f:
    first64 = f.read(64)
    f.seek(max(0, size - 131072))
    tail128k = f.read()

print("=== ARCHIVO LOCAL ===")
print("Primeros 64 bytes HEX:")
print(hb(first64))
print("ASCII:")
print(ascii_preview(first64))
print()
print("Empieza como ZIP PK\\x03\\x04:", first64.startswith(b"PK\\x03\\x04"))
print("EOCD clásico PK\\x05\\x06 en últimos 128 KiB:", b"PK\\x05\\x06" in tail128k)
print("ZIP64 EOCD PK\\x06\\x06 en últimos 128 KiB:", b"PK\\x06\\x06" in tail128k)
print("ZIP64 locator PK\\x06\\x07:", b"PK\\x06\\x07" in tail128k)
print()

print("=== PROBANDO RANGE DEL CDN: bytes 0-63 ===")
rc1, h1, rfirst, err1 = curl_range("0-63")
print("curl rc:", rc1)
print(h1[-2500:])
print("Bytes recibidos:", len(rfirst))
print("HEX:", hb(rfirst[:64]))
print("Coincide con local:", rfirst[:64] == first64[:64])
print()

print("=== PROBANDO RANGE DEL CDN: últimos 64 bytes ===")
rc2, h2, rlast, err2 = curl_range("-64")
print("curl rc:", rc2)
print(h2[-2500:])
print("Bytes recibidos:", len(rlast))
print("HEX:", hb(rlast[-64:]))

local_last64 = tail128k[-64:]
print("Local last64 HEX:", hb(local_last64))
print("Coincide con local:", rlast[-64:] == local_last64)
print()

def status_codes(headers):
    return re.findall(r"HTTP/\\S+\\s+(\\d+)", headers)

def content_ranges(headers):
    return re.findall(r"(?im)^content-range:\\s*(.+)$", headers)

print("=== RESUMEN ===")
print("Status inicio:", status_codes(h1))
print("Content-Range inicio:", content_ranges(h1))
print("Status final:", status_codes(h2))
print("Content-Range final:", content_ranges(h2))
print()

range_ok = (
    "206" in status_codes(h1)
    and "206" in status_codes(h2)
    and bool(content_ranges(h1))
    and bool(content_ranges(h2))
)

print("Servidor honra HTTP Range correctamente:", range_ok)
print()

if not first64.startswith(b"PK\\x03\\x04"):
    print("❌ El archivo local NI SIQUIERA empieza como ZIP.")
    print("   Está contaminado o contiene otra respuesta HTTP.")
elif not (b"PK\\x05\\x06" in tail128k or b"PK\\x06\\x06" in tail128k):
    print("❌ Empieza como ZIP, pero NO tiene directorio central al final.")
    print("   Está truncado o fue ensamblado incorrectamente.")
elif rfirst[:64] != first64[:64] or rlast[-64:] != local_last64:
    print("❌ Inicio/final local no coinciden con el archivo oficial.")
    print("   El archivo persistente quedó corrupto durante resume.")
elif not range_ok:
    print("⚠️ El CDN no está respondiendo a Range como necesitamos.")
    print("   No debemos usar curl -C/--continue-at sobre este servidor.")
else:
    print("✅ Inicio, final y Range parecen correctos.")
    print("   Si unzip aún falla, el daño está en el interior del archivo.")
    print("   Siguiente paso: descarga limpia o verificación por bloques.")